# Phase VI — Head-to-head: Tarozo ordinal patterns vs multiscale level-set geometry

This phase tests the closest interpretable prior representation directly against our level-set geometry on the **same ArtBench pilot** and under the **same artist-disjoint nested-CV protocol** used in Phases IV/IVb.

Tarozo et al. (PNAS Nexus, 2025) represent paintings by the frequencies of **75 two-by-two ordinal patterns that explicitly preserve rank ties**, grouped into 11 interpretable types. Their paper also compares compressed ordinal representations such as the complexity–entropy plane. The current `ordpy>=1.2` package from the same research group implements `two_by_two_patterns`, including the 75-pattern and 11-group representations.

The primary scientific question is:

\[
\boxed{\text{Does multiscale level-set curvature add information beyond tie-aware ordinal patterns?}}
\]

We do **not** compare absolute accuracies with Tarozo et al., because the corpus, style set, and validation protocol differ. Instead, we hold the probe classifier and grouped validation protocol fixed so that the changing factor is the representation.

### Prespecified primary tests

For both ArtBench-10 and the source-controlled WikiArt-8 subset:

1. `OP75 + K40` versus `OP75`;
2. the same contrast after exact `k=40` training-only feature selection.

For WikiArt-8, the stricter contextual test additionally asks whether curvature adds information after **both** the 90-feature conventional appearance baseline and the 75 ordinal probabilities are present.


In [ ]:
import os, sys, subprocess, shutil, zipfile
from pathlib import Path

REPO_URL = "https://github.com/ardominguezm/painting-geometry.git"
BRANCH = "multiscale-corpus-analysis"
REPO_DIR = Path("/content/painting-geometry")

os.chdir("/content")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
# ordpy 1.2 introduced the two_by_two_patterns implementation corresponding to the 2025 paper.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ordpy>=1.2.0"], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import pandas as pd, numpy as np
import ordpy
print("Branch:", BRANCH)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("ordpy:", getattr(ordpy, "__version__", "unknown"))
assert hasattr(ordpy, "two_by_two_patterns"), "ordpy>=1.2.0 is required"


## 1. Upload the Phase-IV and Phase-IVb ZIPs

Upload:

- `painting_geometry_phase4_artbench_pilot.zip` — contains the exact 4,000-image pilot feature matrix and filenames;
- `painting_geometry_phase4b_scale_hierarchy.zip` — contains already-computed artist-disjoint OOF predictions for the 40-feature curvature representation and the conventional baseline.

Reusing the Phase-IVb OOF predictions avoids refitting models that were already run under the same frozen grouped folds.


In [ ]:
from google.colab import files
uploaded = files.upload()

required = [
    "painting_geometry_phase4_artbench_pilot.zip",
    "painting_geometry_phase4b_scale_hierarchy.zip",
]
for name in required:
    if name not in uploaded:
        raise FileNotFoundError(f"Please upload {name}")

INPUT_DIR = Path("/content/phase6_inputs")
if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)
INPUT_DIR.mkdir(parents=True)

for name in required:
    target = INPUT_DIR / name.replace(".zip", "")
    target.mkdir()
    with zipfile.ZipFile(Path("/content") / name) as zf:
        zf.extractall(target)
    print(name, "->", target)

P4_DIR = INPUT_DIR / "painting_geometry_phase4_artbench_pilot"
P4B_DIR = INPUT_DIR / "painting_geometry_phase4b_scale_hierarchy"

FEATURES0 = P4_DIR / "artbench_pilot_features.csv"
OOF_ALL = P4B_DIR / "artbench10_all_phase4b_oof_predictions.csv"
OOF_W8 = P4B_DIR / "artbench10_wikiart8_phase4b_oof_predictions.csv"

for p in [FEATURES0, OOF_ALL, OOF_W8]:
    if not p.exists():
        # Defensive recursive lookup in case ZIP retained a parent directory.
        matches = list(p.parent.rglob(p.name)) if p.parent.exists() else list(INPUT_DIR.rglob(p.name))
        if matches:
            if p == FEATURES0:
                FEATURES0 = matches[0]
            elif p == OOF_ALL:
                OOF_ALL = matches[0]
            else:
                OOF_W8 = matches[0]

print("Phase IV features:", FEATURES0)
print("Phase IVb all10 OOF:", OOF_ALL)
print("Phase IVb WikiArt8 OOF:", OOF_W8)
assert FEATURES0.exists() and OOF_ALL.exists() and OOF_W8.exists()


## 2. Download the 256×256 ArtBench ImageFolder archive

Ordinal patterns require access to the images again. We download only the 256×256 ImageFolder split, not the original-resolution archive collection.

The exact pilot is recovered by `split/style/filename` from the Phase-IV feature matrix, so **no new sampling is performed**.


In [ ]:
import tarfile, kagglehub

DATA_DIR = Path("/content/artbench_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
KAGGLE_HANDLE = "alexanderliao/artbench10"
TAR_NAME = "artbench-10-imagefolder-split.tar"
tar_path = None

candidate_paths = [
    TAR_NAME,
    f"256X256/{TAR_NAME}",
    f"data/256X256/{TAR_NAME}",
    f"ArtBench-10/data/256X256/{TAR_NAME}",
]
for candidate in candidate_paths:
    try:
        print("Trying Kaggle file:", candidate)
        p = Path(kagglehub.dataset_download(KAGGLE_HANDLE, path=candidate))
        if p.exists():
            tar_path = p
            print("Downloaded:", p)
            break
    except Exception as exc:
        print("  not found:", type(exc).__name__)

if tar_path is None:
    tar_path = DATA_DIR / TAR_NAME
    official = "https://artbench.eecs.berkeley.edu/files/artbench-10-imagefolder-split.tar"
    print("Falling back to official ArtBench URL (~1.85 GB)...")
    subprocess.run(["wget", "-c", official, "-O", str(tar_path)], check=True)

EXTRACT_DIR = DATA_DIR / "imagefolder"
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
if not list(EXTRACT_DIR.rglob("train")):
    print("Extracting", tar_path)
    with tarfile.open(tar_path) as tf:
        try:
            tf.extractall(EXTRACT_DIR, filter="data")
        except TypeError:
            tf.extractall(EXTRACT_DIR)
print("Extracted under:", EXTRACT_DIR)


## 3. Extract the exact Tarozo-style ordinal representations

For every pilot image we compute:

- `OP75`: 75 tie-aware two-by-two pattern probabilities;
- `OP11`: the same patterns compressed into 11 groups;
- `OP24`: the standard 24-pattern representation;
- `OP_HC`: normalized permutation entropy \(H\) and statistical complexity \(C\).

The 75-pattern encoding assigns the **same rank to identical pixel intensities**, exactly the methodological distinction emphasized by Tarozo et al. The extraction uses `ordpy.two_by_two_patterns` from the authors' package and `skimage` luminance grayscale.

This cell checkpoints periodically. If the Colab runtime remains alive and the extraction is interrupted, rerunning it does not harm the input matrix.


In [ ]:
RESULTS = REPO_DIR / "results" / "phase6_ordinal_geometry"
RESULTS.mkdir(parents=True, exist_ok=True)
ENRICHED = RESULTS / "artbench_pilot_features_with_ordinal.csv"

if not ENRICHED.exists():
    subprocess.run(
        [
            sys.executable, "-u", "scripts/extract_tarozo_ordinal_features.py",
            "--features", str(FEATURES0),
            "--dataset-root", str(EXTRACT_DIR),
            "--output", str(ENRICHED),
            "--checkpoint-every", "250",
        ],
        check=True,
    )

enriched = pd.read_csv(ENRICHED)
print("Enriched matrix:", enriched.shape)
for prefix in ["ordhc__", "ord11__", "ord24__", "ord75__", "geom__curv__", "base__"]:
    print(prefix, sum(c.startswith(prefix) for c in enriched.columns))

# Sanity checks: each probability representation must sum to one.
for c in ["ordmeta__sum75", "ordmeta__sum11", "ordmeta__sum24"]:
    print(c, enriched[c].min(), enriched[c].max(), enriched[c].mean())
print("Mean tie-pattern mass:", enriched["ordmeta__tie_pattern_mass"].mean())
print("Mean P([0000]):", enriched["ordmeta__type_A_0000"].mean())


## 4. Artist-disjoint head-to-head

All representations are probed with the **same RBF-SVM pipeline**, the same hyperparameter grid, and the same nested `StratifiedGroupKFold` design used previously:

- 5 outer folds, group = artist;
- 3 inner grouped folds for tuning;
- Macro-F1;
- 95% bootstrap intervals resampled at the **artist** level.

Why not simply reproduce Tarozo's XGBoost accuracy? Because this experiment is designed to compare **representations**, not classifier families. Holding the probe fixed isolates whether ordinal patterns and level-set curvature encode redundant or complementary information.

The run writes checkpoint CSVs after every representation so partial results remain inspectable if the runtime is interrupted. A fresh rerun refits the incomplete analysis to preserve the frozen evaluation protocol.


In [ ]:
RUN_DIR = RESULTS / "head_to_head"
RUN_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        sys.executable, "-u", "scripts/run_ordinal_geometry_head_to_head.py",
        "--features", str(ENRICHED),
        "--output-dir", str(RUN_DIR),
        "--phase4b-all-oof", str(OOF_ALL),
        "--phase4b-wiki8-oof", str(OOF_W8),
        "--outer-folds", "5",
        "--inner-folds", "3",
        "--n-jobs", "-1",
        "--metric-boot", "2000",
        "--delta-boot", "5000",
    ],
    check=True,
)


## 5. Inspect the prespecified results

The most important rows are the source-controlled `artbench10_wikiart8` contrasts:

\[
\Delta_{O\to O+K}=F_1(OP75+K40)-F_1(OP75)
\]

and its dimension-matched counterpart. The stricter contextual test asks whether curvature still adds after conventional appearance and ordinal patterns are already present.


In [ ]:
res = pd.read_csv(RUN_DIR / "phase6_head_to_head_results.csv")
delta = pd.read_csv(RUN_DIR / "phase6_head_to_head_deltas.csv")

print("MODEL RESULTS")
display(res[[
    "dataset", "experiment", "n_features_input", "n_features_selected",
    "macro_f1_oof", "macro_f1_ci_low", "macro_f1_ci_high"
]].sort_values(["dataset", "macro_f1_oof"], ascending=[True, False]))

print("\nPAIRED CONTRASTS")
display(delta[[
    "dataset", "contrast", "new_model", "reference", "primary_head_to_head",
    "delta_macro_f1", "delta_ci_low", "delta_ci_high",
    "bootstrap_p_new_gt_ref", "bootstrap_q_bh_primary"
]])

print("\nPRIMARY SOURCE-CONTROLLED TESTS")
display(delta[(delta.dataset == "artbench10_wikiart8") & (delta.primary_head_to_head == True)])


## 6. Package output

Upload the resulting ZIP back to the analysis chat. It contains the ordinal-enriched feature matrix, OOF predictions, fold-level tuning results, paired artist-bootstrap contrasts, and checkpoint tables.


In [ ]:
FINAL_ZIP = Path("/content/painting_geometry_phase6_ordinal_head_to_head.zip")
if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()

# Package the enriched matrix plus all model outputs.
with zipfile.ZipFile(FINAL_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(ENRICHED, arcname=ENRICHED.name)
    fail = ENRICHED.with_suffix(".failures.csv")
    if fail.exists():
        zf.write(fail, arcname=fail.name)
    for p in RUN_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=str(Path("head_to_head") / p.relative_to(RUN_DIR)))

print("ZIP:", FINAL_ZIP, "size MB:", FINAL_ZIP.stat().st_size / 1e6)
files.download(str(FINAL_ZIP))
